# Accident Hotspot Detection using DBSCAN

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import folium
from folium.plugins import HeatMap
import json, os

In [2]:
# Load and sample data
df = pd.read_parquet("../data/accidents_features.parquet")

# Filter: Severity >= 2 (focus on moderate to critical)
df = df[df['Severity'] >= 2]

# Sample max 100,000 rows
df = df.sample(n=min(100000, len(df)), random_state=42)

# Keep only relevant columns
cols = ['Start_Lat', 'Start_Lng', 'Severity', 'hour_of_day', 'weather_risk_score']
df = df[cols].copy()

print(f"Sample size: {len(df)}")

Sample size: 100000


In [3]:
# Scale coordinates for DBSCAN
coords = df[['Start_Lat', 'Start_Lng']].values
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

print(f"Coordinates scaled: {coords_scaled.shape}")

Coordinates scaled: (100000, 2)


In [4]:
# Run DBSCAN clustering
dbscan = DBSCAN(eps=0.01, min_samples=15, n_jobs=-1)
df['cluster'] = dbscan.fit_predict(coords_scaled)

# Count clusters (exclude noise: cluster = -1)
n_clusters = df[df['cluster'] >= 0]['cluster'].nunique()
n_noise = (df['cluster'] == -1).sum()

print(f"Clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")

Clusters found: 98
Noise points: 4712


In [5]:
# Compute cluster statistics
cluster_stats = df[df['cluster'] >= 0].groupby('cluster').agg({
    'Start_Lat': 'mean',
    'Start_Lng': 'mean',
    'Severity': ['mean', 'count'],
    'weather_risk_score': 'mean'
}).reset_index()

cluster_stats.columns = ['cluster', 'lat', 'lng', 'avg_severity', 'accident_count', 'avg_weather_risk']
cluster_stats = cluster_stats.sort_values('accident_count', ascending=False).head(50)

print(f"Top 50 hotspots:\n{cluster_stats.head(10)}")

Top 50 hotspots:
    cluster        lat         lng  avg_severity  accident_count  \
3         3  33.866529 -117.918396      2.507087           20320   
0         0  37.921522 -121.865415      2.390297           11563   
1         1  40.937940  -73.884198      2.437391            6333   
7         7  40.090024  -75.657705      2.147937            5428   
5         5  29.764817  -95.433594      2.205171            5337   
16       16  28.270794  -81.981577      2.240025            4812   
10       10  32.794021  -96.860615      2.387370            4719   
6         6  30.306220  -97.733317      2.132741            3827   
9         9  39.057288  -77.039074      2.564438            3515   
17       17  42.713928  -83.448849      2.453899            3514   

    avg_weather_risk  
3           1.429183  
0           1.500908  
1           1.790147  
7           1.632093  
5           1.694960  
16          1.598712  
10          1.677686  
6           1.654560  
9           1.695875  
17  

In [6]:
# Save cluster data
os.makedirs("../data", exist_ok=True)

# Save CSV
cluster_stats.to_csv("../data/hotspots.csv", index=False)

# Save JSON for dashboard
hotspots_json = cluster_stats.to_dict(orient='records')
with open("../data/hotspots.json", "w") as f:
    json.dump(hotspots_json, f, indent=2)

print("Saved: data/hotspots.csv, data/hotspots.json")

Saved: data/hotspots.csv, data/hotspots.json


In [7]:
# Build Folium Heatmap with CircleMarkers
# Center on US mean coordinates
center_lat = cluster_stats['lat'].mean()
center_lng = cluster_stats['lng'].mean()

m = folium.Map(location=[center_lat, center_lng], zoom_start=4, tiles='cartodbpositron')

# Add heatmap layer from original data
heat_data = df[['Start_Lat', 'Start_Lng', 'Severity']].values.tolist()
HeatMap(heat_data, radius=8, blur=6).add_to(m)

# Add top hotspots as CircleMarkers
for _, row in cluster_stats.head(30).iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=6,
        color='red',
        fill=True,
        popup=f"Accidents: {int(row['accident_count'])}, Avg Severity: {row['avg_severity']:.1f}"
    ).add_to(m)

# Save map
os.makedirs("../data/charts", exist_ok=True)
m.save("../data/hotspot_map.html")
print("Saved: data/charts/hotspot_map.html")

Saved: data/charts/hotspot_map.html


## Hotspot Analysis Summary

- **DBSCAN Clustering**: Identified spatial clusters of accidents using density-based clustering
- **Parameters**: eps=0.01 (scaled), min_samples=15
- **Output Files**:
  - `data/hotspots.csv` - Top 50 hotspot clusters with statistics
  - `data/hotspots.json` - JSON format for dashboard integration
  - `data/charts/hotspot_map.html` - Interactive Folium map with heatmap and markers